#Fine-tuning transformers for Hinglish scam detection

**Before running:** Runtime -> Change runtime type -> **T4 GPU**. Then run cells top to bottom.

We fine-tune two models and compare them with the Day 2 baseline on the *same* train/test split:
1. `distilbert-base-multilingual-cased` (general multilingual model)
2. `l3cube-pune/hing-bert` (BERT pre-trained on romanized Hinglish)

In [1]:
!pip install -q transformers
import torch
print("GPU available:", torch.cuda.is_available())

GPU available: True


## 1. Upload your files
Select **3 files** from your project: `data/clean_messages.csv`, `data/my_test.csv`, `src/preprocess.py`

In [2]:
from google.colab import files
uploaded = files.upload()

Saving clean_messages.csv to clean_messages.csv


## 2. Load data (same split as the baseline: seed 42, 80/20, stratified)

In [4]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from preprocess import clean_text

df = pd.read_csv("clean_messages.csv")
X_tr, X_te, y_tr, y_te = train_test_split(
    df["clean_text"], df["label"], test_size=0.2, stratify=df["label"], random_state=42)

mt = pd.read_csv("my_test.csv")
mt["clean_text"] = mt["text"].apply(clean_text)
print("train:", len(X_tr), "| test:", len(X_te), "| my_test:", len(mt))

train: 349 | test: 88 | my_test: 30


## 3. Training helpers

In [5]:
import time, shutil
from torch.utils.data import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding, set_seed)
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

class MsgDataset(Dataset):
    def __init__(self, texts, labels, tok):
        self.enc = tok(list(texts), truncation=True, max_length=96)
        self.labels = list(labels)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item["labels"] = int(self.labels[i])
        return item

def scores(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "scam_precision": precision_score(y_true, y_pred),
        "scam_recall": recall_score(y_true, y_pred),
        "scam_f1": f1_score(y_true, y_pred),
    }

def run(model_name, epochs=4, lr=3e-5):
    set_seed(42)
    tok = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
    args = TrainingArguments(
        output_dir="out", num_train_epochs=epochs, per_device_train_batch_size=16,
        learning_rate=lr, weight_decay=0.01,
        save_strategy="no", logging_steps=10, report_to="none", seed=42)
    trainer = Trainer(model=model, args=args,
                      train_dataset=MsgDataset(X_tr, y_tr, tok),
                      data_collator=DataCollatorWithPadding(tok))
    t0 = time.time()
    trainer.train()
    print(f"Training time: {time.time() - t0:.0f}s")

    out = {}
    for split, texts, labels in [("test", X_te, y_te), ("my_test", mt["clean_text"], mt["label"])]:
        logits = trainer.predict(MsgDataset(texts, labels, tok)).predictions
        out[split] = scores(labels, np.argmax(logits, axis=1))
    folder = "saved_" + model_name.replace("/", "_")
    model.save_pretrained(folder)
    tok.save_pretrained(folder)
    return out, model, tok, folder

## 4. Train both models (about 2-4 minutes each on a T4)

In [6]:
MODEL_NAMES = ["distilbert-base-multilingual-cased", "l3cube-pune/hing-bert"]
rows, trained = [], {}
for name in MODEL_NAMES:
    print("\n=====", name, "=====")
    try:
        res, model, tok, folder = run(name)
    except Exception as e:
        print("Skipping", name, "->", repr(e))
        import traceback; traceback.print_exc()
        continue
    trained[name] = (model, tok, folder)
    for split in ["test", "my_test"]:
        rows.append({"model": name.split("/")[-1], "split": split, **res[split]})

results = pd.DataFrame(rows).round(3)
results.to_csv("transformer_results.csv", index=False)
results


===== distilbert-base-multilingual-cased =====


config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  542MB            

... (16 outputs left out)

Step,Training Loss
10,0.663115
20,0.546737
30,0.317146
40,0.267061
50,0.136607
60,0.110256
70,0.121666
80,0.125004


Training time: 11s


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

,model,split,accuracy,scam_precision,scam_recall,scam_f1
0,distilbert-base-multilingual-cased,test,0.943,0.950,0.927,0.938
1,distilbert-base-multilingual-cased,my_test,0.967,1.000,0.933,0.966
2,hing-bert,test,0.966,0.932,1.000,0.965
3,hing-bert,my_test,0.967,0.938,1.000,0.968


## 5. Try your own tricky messages

In [7]:
def scam_probability(model, tok, text):
    model.eval()
    enc = tok(clean_text(text), return_tensors="pt", truncation=True, max_length=96).to(model.device)
    with torch.no_grad():
        probs = torch.softmax(model(**enc).logits, dim=-1)[0]
    return float(probs[1])

tests = [
    "Bhai mera phone kho gaya, naye number se msg kar raha hu. 3000 bhej de jaldi",
    "Rs 500 debited from a/c XX1234 at Swiggy. -SBI",
    "Aapka bijli connection aaj kat jayega, call karo 9876500000",
    "Kal shaam ko cricket khelne chalega?",
    "Aapne iPhone jeeta hai! Claim karne ke liye link kholein",
]
for name, (model, tok, _) in trained.items():
    print("\n", name)
    for t in tests:
        print(f"  {scam_probability(model, tok, t):.2f}  {t}")


 distilbert-base-multilingual-cased
  0.17  Bhai mera phone kho gaya, naye number se msg kar raha hu. 3000 bhej de jaldi
  0.01  Rs 500 debited from a/c XX1234 at Swiggy. -SBI
  0.99  Aapka bijli connection aaj kat jayega, call karo 9876500000
  0.01  Kal shaam ko cricket khelne chalega?
  0.06  Aapne iPhone jeeta hai! Claim karne ke liye link kholein

 l3cube-pune/hing-bert
  0.05  Bhai mera phone kho gaya, naye number se msg kar raha hu. 3000 bhej de jaldi
  0.00  Rs 500 debited from a/c XX1234 at Swiggy. -SBI
  0.00  Aapka bijli connection aaj kat jayega, call karo 9876500000
  0.00  Kal shaam ko cricket khelne chalega?
  0.00  Aapne iPhone jeeta hai! Claim karne ke liye link kholein


## 6. Download the best model (optional)
Models are about 500 MB, so only download one. Your Streamlit app can run on the lightweight baseline alone, so this is optional.

In [8]:
assert not results.empty, "No model trained successfully - scroll up to step 4 and read the error"
avg = results.groupby("model")["scam_f1"].mean()
best_short = avg.idxmax()
print("Best by average scam F1:", best_short)
for name, (_, _, folder) in trained.items():
    if name.split("/")[-1] == best_short:
        shutil.make_archive(folder, "zip", folder)
        files.download(folder + ".zip")
files.download("transformer_results.csv")

Best by average scam F1: hing-bert


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>